In [ ]:
!pip install wandb

In [ ]:
import numpy as np
import wandb

from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split

In [ ]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prabhaa-aids2023 (vaishalinir-ymc2022-chennai-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Flatten images
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42
)

In [ ]:
def one_hot_encode(y, num_classes=10):

    one_hot = np.zeros((y.size, num_classes))

    one_hot[np.arange(y.size), y] = 1

    return one_hot


y_train_encoded = one_hot_encode(y_train)
y_val_encoded = one_hot_encode(y_val)
y_test_encoded = one_hot_encode(y_test)

In [ ]:
def sigmoid(z):

    return 1 / (1 + np.exp(-z))


def tanh(z):

    return np.tanh(z)


def relu(z):

    return np.maximum(0, z)


def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
def get_activation(name):

    if name == "sigmoid":
        return sigmoid

    elif name == "tanh":
        return tanh

    elif name == "relu":
        return relu

    else:
        raise ValueError("Invalid activation")

In [ ]:
def __inti__():
  self.activation = get_activation(activation_name)
  A = relu(Z)
  A = self.activation(Z)

In [ ]:
def accuracy(y_true, y_pred):

    predictions = np.argmax(y_pred, axis=1)

    return np.mean(predictions == y_true)

In [ ]:
def train():

    wandb.init()

    config = wandb.config

    # Create hidden layers dynamically
    hidden_layers = [
        config.hidden_size
        for _ in range(config.num_hidden_layers)
    ]

    # Create model
    model = FeedForwardNeuralNetwork(
        input_size=784,
        hidden_layers=hidden_layers,
        output_size=10,
        activation_name=config.activation
    )

    # Training loop
    for epoch in range(config.epochs):

        # Forward pass
        activations, z_values = model.forward(x_train)

        # Loss
        train_loss = cross_entropy_loss(
            y_train_encoded,
            activations[-1]
        )

        # Backpropagation
        gradients_w, gradients_b = model.backward(
            x_train,
            y_train_encoded,
            activations,
            z_values
        )

        # Parameter update
        model.update_parameters(
            gradients_w,
            gradients_b,
            learning_rate=config.learning_rate,
            optimizer=config.optimizer,
            t=epoch + 1
        )

        # Validation
        val_activations, _ = model.forward(x_val)

        val_loss = cross_entropy_loss(
            y_val_encoded,
            val_activations[-1]
        )

        val_accuracy = accuracy(
            y_val,
            val_activations[-1]
        )

        # Log metrics to WandB
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy
        })

In [ ]:
sweep_config = {

    "method": "bayes",

    "metric": {
        "name": "val_accuracy",
        "goal": "maximize"
    },

    "parameters": {

        "epochs": {
            "values": [5, 10]
        },

        "num_hidden_layers": {
            "values": [3, 4, 5]
        },

        "hidden_size": {
            "values": [32, 64, 128]
        },

        "weight_decay": {
            "values": [0, 0.0005, 0.5]
        },

        "learning_rate": {
            "values": [1e-3, 1e-4]
        },

        "optimizer": {
            "values": [
                "sgd",
                "momentum",
                "rmsprop",
                "adam"
            ]
        },

        "batch_size": {
            "values": [16, 32, 64]
        },

        "activation": {
            "values": [
                "sigmoid",
                "tanh",
                "relu"
            ]
        }
    }
}

In [ ]:
sweep_id = wandb.sweep(
    sweep_config,
    project="fashion-mnist-assignment"
)

Create sweep with ID: 5uvbkfc5
Sweep URL: https://wandb.ai/vaishalinir-ymc2022-chennai-institute-of-technology/fashion-mnist-assignment/sweeps/5uvbkfc5


In [ ]:
wandb.agent(
    sweep_id,
    function=train,
    count=20
)

wandb: Agent Starting Run: je2yljos with config:
wandb: 	activation: relu
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 4
wandb: 	optimizer: rmsprop
wandb: 	weight_decay: 0.5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipykernel_5388/1697001774.py", line 14, in train
    model = FeedForwardNeuralNetwork(
            ^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'FeedForwardNeuralNetwork' is not defined



wandb: ERROR Run je2yljos errored: name 'FeedForwardNeuralNetwork' is not defined
wandb: Agent Starting Run: dw17f2cl with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 64
wandb: 	epochs: 5
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 4
wandb: 	optimizer: adam
wandb: 	weight_decay: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipykernel_5388/1697001774.py", line 14, in train
    model = FeedForwardNeuralNetwork(
            ^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'FeedForwardNeuralNetwork' is not defined



wandb: ERROR Run dw17f2cl errored: name 'FeedForwardNeuralNetwork' is not defined
wandb: Agent Starting Run: mhqwtiod with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 64
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 3
wandb: 	optimizer: adam
wandb: 	weight_decay: 0.5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 304, in _run_job
    self._function()
  File "/tmp/ipykernel_5388/1697001774.py", line 14, in train
    model = FeedForwardNeuralNetwork(
            ^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'FeedForwardNeuralNetwork' is not defined



wandb: ERROR Run mhqwtiod errored: name 'FeedForwardNeuralNetwork' is not defined
wandb: ERROR Detected 3 failed runs in the first 60 seconds, killing sweep.
wandb: To disable this check set WANDB_AGENT_DISABLE_FLAPPING=true


In [ ]:
# =========================================================
# TASK 4 - W&B HYPERPARAMETER SWEEP
# COMPLETE WORKING CODE
# =========================================================

# =========================
# INSTALL W&B
# =========================

# Run this once in Colab/Jupyter
# !pip install wandb


# =========================
# IMPORT LIBRARIES
# =========================

import numpy as np
import wandb

from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split


# =========================
# LOGIN TO W&B
# =========================

wandb.login()


# =========================
# LOAD DATASET
# =========================

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()


# =========================
# PREPROCESS DATA
# =========================

# Flatten images
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0


# =========================
# TRAIN VALIDATION SPLIT
# =========================

x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42
)


# =========================
# ONE HOT ENCODING
# =========================

def one_hot_encode(y, num_classes=10):

    one_hot = np.zeros((y.size, num_classes))

    one_hot[np.arange(y.size), y] = 1

    return one_hot


y_train_encoded = one_hot_encode(y_train)
y_val_encoded = one_hot_encode(y_val)
y_test_encoded = one_hot_encode(y_test)


# =========================
# ACTIVATION FUNCTIONS
# =========================

def sigmoid(z):

    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(z):

    s = sigmoid(z)

    return s * (1 - s)


def tanh(z):

    return np.tanh(z)


def tanh_derivative(z):

    return 1 - np.tanh(z) ** 2


def relu(z):

    return np.maximum(0, z)


def relu_derivative(z):

    return (z > 0).astype(float)


def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# =========================
# ACTIVATION SELECTOR
# =========================

def get_activation(name):

    if name == "sigmoid":

        return sigmoid, sigmoid_derivative

    elif name == "tanh":

        return tanh, tanh_derivative

    elif name == "relu":

        return relu, relu_derivative

    else:

        raise ValueError("Invalid activation")


# =========================
# LOSS FUNCTION
# =========================

def cross_entropy_loss(y_true, y_pred):

    epsilon = 1e-10

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    return loss


# =========================
# ACCURACY FUNCTION
# =========================

def accuracy(y_true, y_pred):

    predictions = np.argmax(y_pred, axis=1)

    return np.mean(predictions == y_true)


# =========================
# FEEDFORWARD NEURAL NETWORK
# =========================

class FeedForwardNeuralNetwork:

    def __init__(
        self,
        input_size,
        hidden_layers,
        output_size,
        activation_name="relu"
    ):

        self.layers = [input_size] + hidden_layers + [output_size]

        self.weights = []
        self.biases = []

        self.activation_name = activation_name

        self.activation, self.activation_derivative = (
            get_activation(activation_name)
        )

        # Xavier Initialization
        for i in range(len(self.layers) - 1):

            weight = np.random.randn(
                self.layers[i],
                self.layers[i + 1]
            ) * np.sqrt(1 / self.layers[i])

            bias = np.zeros((1, self.layers[i + 1]))

            self.weights.append(weight)
            self.biases.append(bias)

        # Adam Variables
        self.m_w = [np.zeros_like(w) for w in self.weights]
        self.m_b = [np.zeros_like(b) for b in self.biases]

        self.v_w = [np.zeros_like(w) for w in self.weights]
        self.v_b = [np.zeros_like(b) for b in self.biases]

    # =========================
    # FORWARD PROPAGATION
    # =========================

    def forward(self, X):

        activations = [X]
        z_values = []

        A = X

        # Hidden layers
        for i in range(len(self.weights) - 1):

            Z = np.dot(A, self.weights[i]) + self.biases[i]

            z_values.append(Z)

            A = self.activation(Z)

            activations.append(A)

        # Output layer
        Z = np.dot(A, self.weights[-1]) + self.biases[-1]

        z_values.append(Z)

        output = softmax(Z)

        activations.append(output)

        return activations, z_values

    # =========================
    # BACKPROPAGATION
    # =========================

    def backward(self, X, y, activations, z_values):

        m = X.shape[0]

        gradients_w = []
        gradients_b = []

        # Output error
        dZ = activations[-1] - y

        # Backpropagation
        for i in reversed(range(len(self.weights))):

            dW = np.dot(activations[i].T, dZ) / m

            dB = np.sum(dZ, axis=0, keepdims=True) / m

            gradients_w.insert(0, dW)
            gradients_b.insert(0, dB)

            if i > 0:

                dA = np.dot(dZ, self.weights[i].T)

                dZ = (
                    dA *
                    self.activation_derivative(z_values[i - 1])
                )

        return gradients_w, gradients_b

    # =========================
    # SGD OPTIMIZER
    # =========================

    def sgd(
        self,
        gradients_w,
        gradients_b,
        learning_rate
    ):

        for i in range(len(self.weights)):

            self.weights[i] -= (
                learning_rate * gradients_w[i]
            )

            self.biases[i] -= (
                learning_rate * gradients_b[i]
            )

    # =========================
    # ADAM OPTIMIZER
    # =========================

    def adam(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        t,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            # Momentum
            self.m_w[i] = (
                beta1 * self.m_w[i]
                + (1 - beta1) * gradients_w[i]
            )

            self.m_b[i] = (
                beta1 * self.m_b[i]
                + (1 - beta1) * gradients_b[i]
            )

            # RMSProp
            self.v_w[i] = (
                beta2 * self.v_w[i]
                + (1 - beta2) * (gradients_w[i] ** 2)
            )

            self.v_b[i] = (
                beta2 * self.v_b[i]
                + (1 - beta2) * (gradients_b[i] ** 2)
            )

            # Bias correction
            m_w_hat = self.m_w[i] / (
                1 - beta1 ** t
            )

            m_b_hat = self.m_b[i] / (
                1 - beta1 ** t
            )

            v_w_hat = self.v_w[i] / (
                1 - beta2 ** t
            )

            v_b_hat = self.v_b[i] / (
                1 - beta2 ** t
            )

            # Update
            self.weights[i] -= (
                learning_rate *
                m_w_hat /
                (np.sqrt(v_w_hat) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                m_b_hat /
                (np.sqrt(v_b_hat) + epsilon)
            )

    # =========================
    # UPDATE PARAMETERS
    # =========================

    def update_parameters(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        optimizer="sgd",
        t=1
    ):

        if optimizer == "sgd":

            self.sgd(
                gradients_w,
                gradients_b,
                learning_rate
            )

        elif optimizer == "adam":

            self.adam(
                gradients_w,
                gradients_b,
                learning_rate,
                t
            )

        else:

            self.sgd(
                gradients_w,
                gradients_b,
                learning_rate
            )


# =========================
# TRAIN FUNCTION
# =========================

def train():

    # Initialize WandB
    wandb.init()

    config = wandb.config

    # Run Name
    wandb.run.name = (
        f"hl_{config.num_hidden_layers}"
        f"_bs_{config.batch_size}"
        f"_ac_{config.activation}"
    )

    # Hidden Layers
    hidden_layers = [
        config.hidden_size
        for _ in range(config.num_hidden_layers)
    ]

    # Create Model
    model = FeedForwardNeuralNetwork(
        input_size=784,
        hidden_layers=hidden_layers,
        output_size=10,
        activation_name=config.activation
    )

    # Training Loop
    for epoch in range(config.epochs):

        # Forward Pass
        activations, z_values = model.forward(x_train)

        # Loss
        train_loss = cross_entropy_loss(
            y_train_encoded,
            activations[-1]
        )

        # Backpropagation
        gradients_w, gradients_b = model.backward(
            x_train,
            y_train_encoded,
            activations,
            z_values
        )

        # Parameter Update
        model.update_parameters(
            gradients_w,
            gradients_b,
            learning_rate=config.learning_rate,
            optimizer=config.optimizer,
            t=epoch + 1
        )

        # Validation
        val_activations, _ = model.forward(x_val)

        val_loss = cross_entropy_loss(
            y_val_encoded,
            val_activations[-1]
        )

        val_accuracy = accuracy(
            y_val,
            val_activations[-1]
        )

        # Log to WandB
        wandb.log({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "val_loss": val_loss,

            "val_accuracy": val_accuracy
        })


# =========================
# SWEEP CONFIGURATION
# =========================

sweep_config = {

    "method": "bayes",

    "metric": {

        "name": "val_accuracy",

        "goal": "maximize"
    },

    "parameters": {

        "epochs": {

            "values": [5, 10]
        },

        "num_hidden_layers": {

            "values": [3, 4]
        },

        "hidden_size": {

            "values": [32, 64, 128]
        },

        "learning_rate": {

            "values": [1e-3, 1e-4]
        },

        "optimizer": {

            "values": [
                "sgd",
                "adam"
            ]
        },

        "batch_size": {

            "values": [16, 32]
        },

        "activation": {

            "values": [
                "relu",
                "tanh",
                "sigmoid"
            ]
        }
    }
}


# =========================
# CREATE SWEEP
# =========================

sweep_id = wandb.sweep(

    sweep_config,

    project="fashion-mnist-assignment"
)


# =========================
# RUN SWEEP AGENT
# =========================

wandb.agent(

    sweep_id,

    function=train,

    count=5
)

Create sweep with ID: 5545h2vp
Sweep URL: https://wandb.ai/vaishalinir-ymc2022-chennai-institute-of-technology/fashion-mnist-assignment/sweeps/5545h2vp


wandb: Agent Starting Run: f3pf45av with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 16
wandb: 	epochs: 5
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 4
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁▃▅▆█
train_loss,█▆▄▃▁
val_accuracy,▁▁▁▁▁
val_loss,█▆▄▃▁
epoch,5
train_loss,2.6511
val_accuracy,0.1045
val_loss,2.63571


wandb: Agent Starting Run: ppsfm98x with config:
wandb: 	activation: relu
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0001
wandb: 	num_hidden_layers: 3
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▆▆▅▄▃▃▂▁
val_accuracy,▁▁▁▁▁█████
val_loss,█▇▆▆▅▄▃▃▂▁
epoch,10
train_loss,2.28924
val_accuracy,0.10067
val_loss,2.28939


...

epoch,▁▃▅▆█
train_loss,█▆▄▃▁
val_accuracy,▁▃▅▇█
val_loss,█▆▄▂▁
epoch,5
train_loss,2.29216
val_accuracy,0.13583
val_loss,2.28764


wandb: Agent Starting Run: 92vkpsvl with config:
wandb: 	activation: tanh
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 4
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▆▆▅▄▃▃▂▁
val_accuracy,▁▂▁▂▃▄▄▆██
val_loss,█▇▆▆▅▄▃▃▂▁
epoch,10
train_loss,2.3364
val_accuracy,0.11083
val_loss,2.33574


wandb: Agent Starting Run: b86n8vyc with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 32
wandb: 	epochs: 5
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.001
wandb: 	num_hidden_layers: 4
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁▃▅▆█
train_loss,█▆▄▂▁
val_accuracy,▁▁▁▁▁
val_loss,█▆▄▂▁
epoch,5
train_loss,2.32951
val_accuracy,0.09833
val_loss,2.31536


# Task 4: Hyperparameter Tuning using Weights & Biases (WandB)

## Objective
The objective of this task is to perform hyperparameter tuning for the feedforward neural network implemented from scratch using NumPy. The tuning process is automated using the sweep functionality provided by Weights & Biases (WandB).

The goal is to identify the best set of hyperparameters that maximize validation accuracy on the Fashion-MNIST dataset.

---

# Dataset
The Fashion-MNIST dataset is used for experimentation.

Dataset details:
- 60,000 training images
- 10,000 testing images
- 10 output classes
- Image size: 28 × 28 pixels
- Flattened input dimension: 784

The standard train-test split provided by:
```python id="f9rjz2"
fashion_mnist.load_data()